# 06 · MCP – plug J.A.R.V.I.S. into any toolbox

The **Model Context Protocol (MCP)** is an open standard for exposing tools to AI. An MCP server lists tools (name, description, schema); an MCP client (here: Foundry Agent Service) lets the model call them.

| Setting on the `mcp` tool | Purpose |
|---|---|
| `server_label` | unique name of the server inside the agent |
| `server_url` | remote MCP endpoint (must be reachable from Foundry) |
| `allowed_tools` | allow-list of tool names (default: all) |
| `require_approval` | `always` (default) \| `never` \| per-tool lists |
| `project_connection_id` | project connection holding credentials |

Deep dive: [docs/mcp-skills.md](../docs/mcp-skills.md).

In [ ]:
import sys; sys.path.insert(0, '.')
from stark_helpers import *
from stark_replies import show_reply, mcp_calls

from azure.ai.projects.models import PromptAgentDefinition, MCPTool
LEARN_URL = "https://learn.microsoft.com/api/mcp"
OPS_AGENT_NAME = "stark-ops"     # deployed from agents/stark-ops.yaml in section 2
created_agents = []              # agent names, deleted in the clean-up

## 1 · A public MCP server: Microsoft Learn
No credentials needed. `require_approval="always"` means **you** see every tool call before it runs. `ask_agent()` from [stark_helpers.py](stark_helpers.py) answers each `mcp_approval_request` with your `approve(item)` function and prints one line per request; `show_reply()` from [stark_replies.py](stark_replies.py) then shows the prompt, the whole answer and the tool calls.

### Case A · Approve the expected server
**Goal:** see each MCP call before it runs, and let through only calls to the server we expect.

**Expected:** one or more `approval request: learn.<tool>(...) -> APPROVED` lines, then a two-sentence answer that cites Microsoft Learn page titles. Which Learn tool the model picks is ❓ (most likely `microsoft_docs_search`).

In [ ]:
learn_tool = MCPTool(server_label="learn", server_url=LEARN_URL, require_approval="always")
learn_agent = project().agents.create_version(
    agent_name="jarvis-nb06-learn",
    definition=PromptAgentDefinition(model=model("chat"), instructions="Use the MCP tools to answer. Cite the page titles.",
                                     tools=[learn_tool]),
)
created_agents.append(learn_agent.name)

conversation_question = "How do I create a conversation with the Foundry Agent Service in Python? Two sentences."
approved_reply = ask_agent(learn_agent, conversation_question,
                           approve=lambda item: item.server_label == "learn")   # approve only the server we expect
show_reply(conversation_question, approved_reply)

### Case B · Approve nothing
**Goal:** the same agent and question, one change: no `approve` function. `ask_agent()` then denies every request, the safe default.

**Expected:** one or more `-> DENIED` lines and no `mcp_call` in the tool calls. ❓ The model then answers from its own knowledge without page titles, or says it could not look it up.

In [ ]:
denied_reply = ask_agent(learn_agent, conversation_question)        # no approve=...: every request is denied
show_reply(conversation_question, denied_reply, checks={"MCP calls ran": ", ".join(mcp_calls(denied_reply)) or "none"})

### Restrict the tools
A server may expose dozens of tools. `allowed_tools` is an allow-list: the model never even sees the others. Use it and `require_approval` for anything that writes data.

But first **look at what the server offers**. An MCP server describes its tools to any client, so we can ask it directly (no agent, no model call) and then decide which names go on the allow-list.

**Expected:** three tools at the time of writing: `microsoft_docs_search`, `microsoft_docs_fetch` and `microsoft_code_sample_search`, each with its parameters and the first line of its description.

In [ ]:
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

async with streamablehttp_client(LEARN_URL) as (read, write, _):
    async with ClientSession(read, write) as session:
        await session.initialize()
        server_tools = (await session.list_tools()).tools

print(len(server_tools), "tools offered by the server")
for t in server_tools:
    first_line = (t.description or "").strip().splitlines()[0]
    print(f"- {t.name}({', '.join(t.inputSchema.get('properties', {}))}): {first_line[:90]}")

All three are read-only (search and fetch). For a server with write tools, copy only the names you need into `allowed_tools` and keep `require_approval="always"` for any tool that changes data.

### Case C · Allow one tool, no approval
**Goal:** a new version of the same agent that sees only `microsoft_docs_search` and runs it without asking.

**Expected:** no `approval request` lines, only `microsoft_docs_search` in the MCP calls ✅, and a one-sentence answer.

Calls without a version use the latest, so from here on `jarvis-nb06-learn` asks for no approvals. To repeat case B, run case A first: it creates a new version with `require_approval="always"`.

In [ ]:
search_only_tool = MCPTool(server_label="learn", server_url=LEARN_URL,
                           allowed_tools=["microsoft_docs_search"], require_approval="never")
learn_agent_v2 = project().agents.create_version(
    agent_name="jarvis-nb06-learn",
    definition=PromptAgentDefinition(model=model("chat"), instructions="Use the MCP tools to answer. One sentence.",
                                     tools=[search_only_tool]),
)
endpoint_question = "What is a Foundry project endpoint?"
search_reply = ask_agent(learn_agent_v2, endpoint_question)
used = mcp_calls(search_reply)
show_reply(endpoint_question, search_reply, checks={
    "MCP calls": ", ".join(used) or "none",
    "only microsoft_docs_search": mark(set(used) <= {"microsoft_docs_search"}),
})

## 2 · Our own MCP server: `stark-mcp`
[src/stark-mcp](../src/stark-mcp/function_app.py) is an **Azure Function** with the Functions MCP extension: four tools (`list_suits`, `get_suit`, `get_workshop_status`, `check_invoice_approval`). `azd up` deployed it, and the postdeploy hook created the project connection `stark-mcp` that holds the function key – **the key never appears in the agent definition**.

Check that the pieces exist:

In [ ]:
mcp_url = cfg("STARK_MCP_URL")
print("MCP endpoint:", mcp_url)
try:
    conn = project().connections.get("stark-mcp")
    print("connection  :", conn.name, "-", getattr(conn, "type", ""))
except Exception as exc:
    print("connection 'stark-mcp' missing:", type(exc).__name__, "-> run: python scripts/connect_mcp.py")

Deploy the agent from YAML ([agents/stark-ops.yaml](../agents/stark-ops.yaml); chapter 08 explains the script). It allows three of the four tools (`list_suits`, `get_suit`, `get_workshop_status`) with `require_approval: never`, so `ask_agent()` sees no approval requests.

In [ ]:
sh("python scripts/deploy_agents.py agents/stark-ops.yaml")

### Case D · Which suits still work?
**Goal:** the agent answers from our server's data, not from its own idea of Iron Man.

**Expected:** a `list_suits` call (or one `get_suit` per suit) and the answer that **none** is operational: in [function_app.py](../src/stark-mcp/function_app.py) the Mark III is in the museum, the Mark VII and Mark XLII are destroyed and the Mark LXXXV is retired.

In [ ]:
suits_question = "Which suits are still operational?"
suits_reply = ask_agent(OPS_AGENT_NAME, suits_question)
show_reply(suits_question, suits_reply, checks={"MCP calls": ", ".join(mcp_calls(suits_reply)) or "none"})

### Case E · A workshop system
**Goal:** same agent, a different tool.

**Expected:** a `get_workshop_status` call for `fabrication bay` and the answer: busy, with the Mark 86 gauntlet and a drone swarm test in the queue, about 6 hours to go.

In [ ]:
bay_question = "What is the status of the fabrication bay?"
bay_reply = ask_agent(OPS_AGENT_NAME, bay_question)
show_reply(bay_question, bay_reply, checks={
    "MCP calls": ", ".join(mcp_calls(bay_reply)) or "none",
    "called get_workshop_status": mark("get_workshop_status" in mcp_calls(bay_reply)),
})

## 3 · Trust: treat tool output as untrusted
Descriptions and results of a remote MCP server flow straight into the model's context. A hostile or compromised server can embed instructions ("ignore previous instructions...") – **indirect prompt injection**. Defences used here: allow-lists, approvals for risky tools, least-privilege connections. Chapter 14 attacks this on purpose.

## 👀 In the portal
**Build → Agents → jarvis-nb06-learn**: the MCP tool on the agent. The project connection `stark-mcp` is the one `azd` created.

> 💡 All pages: [docs/portal-tour.md](../docs/portal-tour.md)

## 🧹 Clean up

In [ ]:
for name in created_agents + [OPS_AGENT_NAME]:
    delete_agent(name)

## 🎯 Recap
MCP gives agents a **standard plug**: one server, many agents, any client. Remote only – a local `stdio` server must be hosted (Functions, Container Apps) first.

> 💡 Next: [07 · Skills](07_agent_skills.ipynb) 🧪 – reusable know-how.